# Multivariate Imputation

## KNN IMPUTER

In [2]:
import numpy as np
import matplotlib.pyplot as plt 
import pandas as pd 
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer,KNNImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


In [3]:
df = pd.read_csv("DATASET/titanic.csv",usecols=['Age','Pclass','Fare','Survived'])
df

,Survived,Pclass,Age,Fare
0,0,3,22.0,7.2500
1,1,1,38.0,71.2833
2,1,3,26.0,7.9250
3,1,1,35.0,53.1000
4,0,3,35.0,8.0500
...,...,...,...,...
886,0,2,27.0,13.0000
887,1,1,19.0,30.0000
888,0,3,NaN,23.4500
889,1,1,26.0,30.0000


In [4]:
# null percentage 
df.isnull().mean()*100

Survived     0.00000
Pclass       0.00000
Age         19.86532
Fare         0.00000
dtype: float64

In [5]:
X = df.drop(columns=['Survived'])
y = df['Survived']

In [6]:
# spilt the dataset
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.25,random_state=42)

In [7]:
X_train.head(2)

,Pclass,Age,Fare
298,1,NaN,30.50
884,3,25.0,7.05


In [8]:
# apply KNN imputer
knn  = KNNImputer(n_neighbors=5,weights='distance')
X_train_knn =knn.fit_transform(X_train)
X_test_knn  = knn.transform(X_test)

In [9]:
# train a model
model = LogisticRegression()
model.fit(X_train_knn,y_train)
y_pred  = model.predict(X_test_knn)

acc_score = accuracy_score(y_test,y_pred)
print(acc_score)

0.726457399103139


In [10]:
# apply Simple Imputer
si = SimpleImputer()
X_train_si = si.fit_transform(X_train)
X_test_si = si.transform(X_test)

model = LogisticRegression()
model.fit(X_train_si,y_train)
y_pred  = model.predict(X_test_si)

acc_score = accuracy_score(y_test,y_pred)
print(acc_score)

0.7354260089686099


## Iterative Imputer(MICE)

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [12]:
df = np.round(pd.read_csv("DATASET/50_Startups.csv")[['R&D Spend','Administration','Marketing Spend','Profit']]/10000)
np.random.seed(9)
df = df.sample(5)
df

,R&D Spend,Administration,Marketing Spend,Profit
21,8.0,15.0,30.0,11.0
37,4.0,5.0,20.0,9.0
2,15.0,10.0,41.0,19.0
14,12.0,16.0,26.0,13.0
44,2.0,15.0,3.0,7.0


In [13]:
df= df.iloc[:,0:-1]
df

,R&D Spend,Administration,Marketing Spend
21,8.0,15.0,30.0
37,4.0,5.0,20.0
2,15.0,10.0,41.0
14,12.0,16.0,26.0
44,2.0,15.0,3.0


In [14]:
# yaha pe jitne column mai nan values hai utne column slice karo
df.iloc[1,0] = np.nan
df.iloc[3,1] = np.nan
df.iloc[-1,-1] = np.nan

/tmp/ipykernel_9107/188381184.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.iloc[1,0] = np.nan
/tmp/ipykernel_9107/188381184.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.iloc[3,1] = np.nan
/tmp/ipykernel_9107/188381184.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.iloc[-1,-1] = np.nan


In [15]:
df.head(4)

,R&D Spend,Administration,Marketing Spend
21,8.0,15.0,30.0
37,NaN,5.0,20.0
2,15.0,10.0,41.0
14,12.0,NaN,26.0


In [16]:
# impute  all the missing values with mean of respective column
df_m =pd.DataFrame()
df_m['R&D Spend'] = df['R&D Spend'].fillna(df['R&D Spend'].mean())
df_m['Administration'] = df['Administration'].fillna(df['Administration'].mean())
df_m['Marketing Spend'] = df['Marketing Spend'].fillna(df['Marketing Spend'].mean())

In [17]:
# now 0 th iteration : all nan values get fillout by mean
df_m

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,9.25,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.25,26.00
44,2.00,15.00,29.25


In [18]:
# now remove the imputed value from column 1
df_1 =df_m.copy()
df_1.iloc[1,0] = np.nan
df_1

,R&D Spend,Administration,Marketing Spend
21,8.0,15.00,30.00
37,NaN,5.00,20.00
2,15.0,10.00,41.00
14,12.0,11.25,26.00
44,2.0,15.00,29.25


In [19]:
# now we have to build the model to predict the nan value in col 1
# so that all other rows will be our training data and we apply the any ml algo
# to predict the value
X = df_1.iloc[[0,2,3,4],1:3]
X

,Administration,Marketing Spend
21,15.00,30.00
2,10.00,41.00
14,11.25,26.00
44,15.00,29.25


In [21]:
# now make remaing col which has nan value as target column
y =df_1.iloc[[0,2,3,4],0]
y

21     8.0
2     15.0
14    12.0
44     2.0
Name: R&D Spend, dtype: float64

In [23]:
# now apply linear reg on it 
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X,y)

# predict the nan value using model
model.predict(df_1.iloc[1,1:].values.reshape(1,2))

/home/tushar/Desktop/MLandDL/venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([23.14158651])

In [26]:
# add predicted value to the data
df_1.iloc[1,0] = 23.14

In [27]:
df_1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,23.14,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.25,26.00
44,2.00,15.00,29.25


In [32]:
# now remove the imputed value from column 2 
df_1.iloc[3,1] = np.nan

df_1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.0,30.00
37,23.14,5.0,20.00
2,15.00,10.0,41.00
14,12.00,NaN,26.00
44,2.00,15.0,29.25


In [33]:
# now there is  nan value in column 2 at row at index three
# so there are 5 rows in total
# we can use three rows to train model and use first for prediction
X = df_1.iloc[[0,1,2,4],[0,2]]
X

,R&D Spend,Marketing Spend
21,8.00,30.00
37,23.14,20.00
2,15.00,41.00
44,2.00,29.25


In [35]:
y = df_1.iloc[[0,1,2,4],1]
y

21    15.0
37     5.0
2     10.0
44    15.0
Name: Administration, dtype: float64

In [36]:
model = LinearRegression()
model.fit(X,y)
model.predict(df_1.iloc[3,[0,2]].values.reshape(1,2))

/home/tushar/Desktop/MLandDL/venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([11.06331285])

In [38]:
df_1.iloc[3,1] = 11.06

In [39]:
df_1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,23.14,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.06,26.00
44,2.00,15.00,29.25


In [41]:
# now similary for column 3
df_1.iloc[4,-1] = np.nan
# Use last 3 rows to build a model and use the first for prediction
X = df_1.iloc[0:4,0:2]
y = df_1.iloc[0:4,-1]

model = LinearRegression()
model.fit(X,y)

model.predict(df_1.iloc[4,0:2].values.reshape(1,2))

/home/tushar/Desktop/MLandDL/venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([31.56351448])

In [43]:
df_1.iloc[4,-1] = 31.56
df_1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,23.14,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.06,26.00
44,2.00,15.00,31.56


In [44]:
# after first iteration
df_1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,23.14,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.06,26.00
44,2.00,15.00,31.56


In [47]:
# subtract 0th iteration from 1st iteration
df_1 - df_m

,R&D Spend,Administration,Marketing Spend
21,0.00,0.00,0.00
37,13.89,0.00,0.00
2,0.00,0.00,0.00
14,0.00,-0.19,0.00
44,0.00,0.00,2.31


In [48]:
# sklearn implementation of iterative imputer 
from sklearn.experimental import  enable_iterative_imputer
from sklearn.impute import IterativeImputer

imputer = IterativeImputer(max_iter=10,random_state=42)
im_array =imputer.fit_transform(df)

df_imputed = pd.DataFrame(im_array, columns=df.columns)
df_imputed

,R&D Spend,Administration,Marketing Spend
0,8.000000,15.000000,30.000000
1,10.712222,5.000000,20.000000
2,15.000000,10.000000,41.000000
3,12.000000,6.328063,26.000000
4,2.000000,15.000000,12.990453
